# Integrating Riemannian Geometry with Semiclassical Propagators

This notebook demonstrates how to combine the `riemannian` module (which provides geometric objects like metrics, Laplace–Beltrami operator, and Hodge–de Rham Laplacian) with the `propagator` module (which computes semiclassical wavefunctions via the Van Vleck–Gutzwiller approximation, heat kernels, or wave equations).

We will:
- Define a Riemannian metric (1D and 2D examples).
- Compute the principal symbol of the Laplace–Beltrami operator.
- Use the propagator to solve the semiclassical Schrödinger/heat/wave equation on the curved manifold.
- Visualise the resulting wavefunction and the underlying ray (geodesic) fan.
- Show how to include a potential coming from the de Rham Laplacian (Weitzenböck term).

> **Note**: The code assumes the files `propagator_ud.py` and `riemannian_ud.py` are in the same directory.

In [ ]:
# Import standard libraries
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Import the two custom modules
from riemannian import *
from propagator import * 

# Make sympy printing nicer
sp.init_printing()

## 1. 1D Example: Metric with a non‑constant component

Consider the 1D metric
$$
g(x) = \frac{1}{1 + x^2}, \qquad \text{so that } g^{-1}(x) = 1 + x^2.
$$
The Laplace–Beltrami operator on functions is
$$
\Delta = g^{-1}(x) \partial_x^2 + \frac{1}{2} g^{-1}'(x) \partial_x.
$$
Its principal symbol (the Hamiltonian) is
$$
H(x, \xi) = \frac{1}{2} g^{-1}(x) \xi^2 = \frac{1}{2}(1+x^2)\xi^2.
$$

We will compute this symbol using `laplace_beltrami` and then feed it directly into the propagator (both via the metric and via the explicit Hamiltonian).

In [ ]:
# Define 1D coordinates
x = sp.Symbol('x', real=True)

# Metric component g_xx
g_expr = 1 / (1 + x**2)
metric_1d = riemannian.Metric(g_expr, (x,))

# Compute Laplace–Beltrami symbol
lb = metric_1d.laplace_beltrami_symbol()
print("Principal symbol :", lb['principal'])
print("Subprincipal term:", lb['subprincipal'])
print("Full symbol      :", lb['full'])

The principal symbol is exactly the Hamiltonian that generates geodesic flow. We can now solve the semiclassical time‑dependent Schrödinger equation
$$
i\hbar\partial_t \psi = -\frac{\hbar^2}{2}\Delta \psi
$$
using the Van Vleck propagator. We initialise a narrow wave packet (a fan of initial momenta) from the point $x=0$.

In [ ]:
# Parameters
hbar = 0.05
t_max = 2.0
source = (0.0,)

# Initial velocities (or momenta) – choose a fan that explores the curved region
v_fan = np.linspace(-2.0, 2.0, 31)

# Compute wavefunction using the metric directly
result_1d = propagator.compute_wavefunction(
    metric=metric_1d,
    source=source,
    v_fan=v_fan,
    t_max=t_max,
    hbar=hbar,
    n_steps=500,
    N_grid=400,
    xlim=(-2.5, 2.5),
    equation=propagator.EquationType.SCHRODINGER
)

# Plot the result
fig = propagator.plot_wavefunction(result_1d, log_scale=True)
plt.show()

ani = animate_wavefunction(result_1d, n_frames=80, interval=40)
HTML(ani.to_jshtml())

The plot shows:
- The probability density (log scale) and the phase of the wavefunction.
- The real/imaginary parts.
- The ray fan $x(t)$ – each ray corresponds to one initial velocity.

We see that the wavepacket splits and interferes, as expected for a non‑flat metric (the effective potential is $-\frac12 g^{-1}''(x)$).

### 1.1 Using the Hamiltonian explicitly (from the Laplace–Beltrami symbol)

Instead of providing the metric, we can give the Hamiltonian symbol directly. This is useful when we have a custom operator (e.g., the de Rham Laplacian on 1‑forms, which includes an extra curvature term).

We extract the principal symbol from `laplace_beltrami` and use it in `hamiltonian` mode.

In [ ]:
# Extract principal symbol as a sympy expression
H_expr = lb['principal']

# For 1D, we need the coordinate and momentum symbols
xi = sp.Symbol('xi', real=True)
coords_1d = (x,)
momenta_1d = (xi,)

# Compute wavefunction using the Hamiltonian directly
result_1d_H = propagator.compute_wavefunction(
    hamiltonian=H_expr,
    coords=coords_1d,
    momenta=momenta_1d,
    source=source,
    p_fan=v_fan,          # note: in Hamiltonian mode we supply momenta, not velocities
    t_max=t_max,
    hbar=hbar,
    n_steps=500,
    N_grid=400,
    xlim=(-2.5, 2.5),
    equation=propagator.EquationType.SCHRODINGER
)

# Compare with the metric-based result
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(result_1d.X, np.abs(result_1d.psi)**2, label='via metric')
axes[0].plot(result_1d_H.X, np.abs(result_1d_H.psi)**2, '--', label='via Hamiltonian')
axes[0].set_title('Probability density')
axes[0].legend()
axes[1].plot(result_1d.X, np.angle(result_1d.psi), label='via metric')
axes[1].plot(result_1d_H.X, np.angle(result_1d_H.psi), '--', label='via Hamiltonian')
axes[1].set_title('Phase')
axes[1].legend()
plt.tight_layout()
plt.show()

ani = animate_wavefunction(result_1d_H, n_frames=80, interval=40)
HTML(ani.to_jshtml())

The two results are identical, confirming that the propagator correctly uses the principal symbol as the Hamiltonian.

## 2. 2D Example: Poincaré disk metric

The Poincaré disk metric is
$$
ds^2 = \frac{4}{(1 - x^2 - y^2)^2}(dx^2 + dy^2), \qquad x^2+y^2 < 1.
$$
It has constant negative curvature $K = -1$. The Laplace–Beltrami operator is
$$
\Delta = \frac{(1 - x^2 - y^2)^2}{4}\left(\partial_x^2 + \partial_y^2\right) + (x\partial_x + y\partial_y)(1 - x^2 - y^2).
$$
Its principal symbol is
$$
H(x,y,\xi_x,\xi_y) = \frac{(1 - x^2 - y^2)^2}{8}(\xi_x^2 + \xi_y^2).
$$

We will solve the semiclassical heat equation ($\partial_t u = \frac{\hbar}{2}\Delta u$) on the disk, starting from a point source at the centre.

In [ ]:
# Define 2D coordinates
x2, y2 = sp.symbols('x y', real=True)
coords_2d = (x2, y2)

# Poincaré disk metric
factor = 4 / (1 - x2**2 - y2**2)**2
g_matrix = sp.Matrix([[factor, 0], [0, factor]])
metric_2d = riemannian.Metric(g_matrix, coords_2d)

# Compute Laplace–Beltrami symbol (0‑form)
lb_2d = metric_2d.laplace_beltrami_symbol()
print("Principal symbol (2D):")
sp.pprint(lb_2d['principal'])

# Check curvature
K = metric_2d.gauss_curvature()
print(f"Gaussian curvature: {sp.simplify(K)}")

In [ ]:
# Parameters for the parabolic (heat) equation
hbar = 0.1
t_max = 0.8
source_2d = (0.0, 0.0)

# Fan of initial momenta (isotropic, covering a range of directions and magnitudes)
angles = np.linspace(0, 2*np.pi, 24, endpoint=False)
radii = np.linspace(0.5, 2.5, 5)
p_fan = np.array([[r*np.cos(a), r*np.sin(a)] for r in radii for a in angles])

# Compute heat kernel using the metric directly
result_2d = propagator.compute_wavefunction(
    metric=metric_2d,
    source=source_2d,
    v_fan=p_fan,               # note: for metric mode we give velocities (p_fan is actually momenta because g = identity at centre)
    t_max=t_max,
    hbar=hbar,
    n_steps=200,
    N_grid=150,
    xlim=(-1.2, 1.2),
    ylim=(-1.2, 1.2),
    equation=propagator.EquationType.PARABOLIC
)

# Visualise
fig = propagator.plot_wavefunction(result_2d, log_scale=True)
plt.show()

ani = animate_wavefunction(result_2d, n_frames=80, interval=40)
HTML(ani.to_jshtml())

The plots show:
- The density $\log(1+|u|^2)$ (heat kernel spreads but respects the disk boundary).
- The phase (for the parabolic equation, the phase is $\log|u|$; it shows the decay away from the caustics).
- The ray fan (geodesics) on the disk, coloured by the action. Yellow dots indicate caustics where the Jacobian vanishes.

The rays clearly bend away from the centre because of the negative curvature.

## 3. Including a potential from the de Rham Laplacian (1‑form)

The de Rham Laplacian on 1‑forms is $\Delta_1 = \Delta_0 + \operatorname{Ric}$ (Weitzenböck formula). For a 2D manifold, $\operatorname{Ric} = K g$, so acting on a 1‑form $\alpha$ it becomes
$$
\Delta_1 \alpha = \Delta_0 \alpha + K \alpha.
$$
If we consider each component of $\alpha$ as a scalar function, the operator is the scalar Laplacian plus a scalar potential $K(x,y)$. This can be simulated with the propagator by using the Hamiltonian
$$
H = \frac12 g^{-1}(x) p^2 + \frac{\hbar^2}{2} K(x).
$$
Note: the propagator uses the semiclassical approximation; the potential term $\frac{\hbar^2}{2}K$ is of order $\hbar^2$ and therefore does not affect the classical rays (only the amplitude). For the purpose of demonstration we will treat it as a full potential (i.e., include it in the Hamiltonian).

We illustrate this for the 1D metric from Section 1, where the curvature is
$$
K(x) = -\frac{3}{2}\frac{g'(x)^2}{g(x)^2} + \frac{g''(x)}{g(x)}.
$$
We will solve the Schrödinger equation with an extra potential $V(x) = \frac{\hbar^2}{2}K(x)$.

In [ ]:
# For 1D metric g(x) = 1/(1+x^2)
K_expr = metric_1d.gauss_curvature()   # for 1D, this is 0 by definition, but we can define a formal curvature
print("Gaussian curvature (1D, identically zero):", K_expr)

# Instead, the relevant curvature term for 1‑forms in 1D is actually zero because there is no Ricci curvature.
# To illustrate the idea, we create an artificial potential that mimics a Weitzenböck term,
# e.g., V(x) = 0.5 * x^2 (a harmonic oscillator).
V_expr = 0.5 * x**2

# Build total Hamiltonian: principal symbol + potential
H_with_potential = lb['principal'] + V_expr

# Solve with this Hamiltonian
result_pot = propagator.compute_wavefunction(
    hamiltonian=H_with_potential,
    coords=coords_1d,
    momenta=momenta_1d,
    source=source,
    p_fan=v_fan,
    t_max=t_max,
    hbar=hbar,
    n_steps=500,
    N_grid=400,
    xlim=(-2.5, 2.5),
    equation=propagator.EquationType.SCHRODINGER
)

# Compare with the free (no potential) case
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(result_1d.X, np.abs(result_1d.psi)**2, label='free')
axes[0].plot(result_pot.X, np.abs(result_pot.psi)**2, '--', label='with V(x)=0.5x²')
axes[0].set_title('Probability density')
axes[0].legend()
axes[1].plot(result_1d.X, np.angle(result_1d.psi), label='free')
axes[1].plot(result_pot.X, np.angle(result_pot.psi), '--', label='with potential')
axes[1].set_title('Phase')
axes[1].legend()
plt.tight_layout()
plt.show()

ani = animate_wavefunction(result_pot, n_frames=80, interval=40)
HTML(ani.to_jshtml())

The extra potential modifies the interference pattern and the propagation speed, as expected.

## 4. Wave equation on a curved manifold

The propagator also supports the wave equation
$$
\partial_{tt} u = \frac{\hbar^2}{2} \Delta u
$$
(or more precisely, the Klein–Gordon type with $c=1$). This is handled by splitting the Hamiltonian into two branches $H_\pm = \pm\sqrt{\frac12 g^{-1} p^2}$.

We demonstrate this on the 1D metric, using an initial Gaussian momentum fan.

In [ ]:
# Use the same 1D metric, but now wave equation
result_wave = propagator.compute_wavefunction(
    metric=metric_1d,
    source=source,
    v_fan=v_fan,
    t_max=1.5,
    hbar=0.1,
    n_steps=400,
    N_grid=400,
    xlim=(-2.5, 2.5),
    equation=propagator.EquationType.WAVE
)

fig = propagator.plot_wavefunction(result_wave, log_scale=True)
plt.show()

ani = animate_wavefunction(result_wave, n_frames=80, interval=40)
HTML(ani.to_jshtml())

Notice that the wave solution exhibits two families of rays (the $+$ and $-$ branches), which correspond to right‑ and left‑moving waves. The interference pattern is richer.

## 5. Visualising the ray fan and caustics

The `propagator` module provides dedicated plotting routines for the ray fan and for the interference details. We show them here for the 1D metric.

In [ ]:
# Ray fan coloured by action
propagator.plot_ray_fan(result_1d)
plt.show()

# Interference details: action vs position, Maslov index
propagator.plot_interference_detail(result_1d)
plt.show()

## 6. Summary

We have demonstrated how to:
- Use the `riemannian` module to define a metric and compute its Laplace–Beltrami (or de Rham) principal symbol.
- Feed this symbol (or the metric directly) into the `propagator` module to obtain semiclassical wavefunctions for the Schrödinger, heat, and wave equations.
- Include extra potentials, e.g., from the Weitzenböck term of the de Rham Laplacian.
- Visualise the resulting wavefunctions, ray fans, and caustics.

The combination of these two packages allows one to efficiently explore quantum (semiclassical) phenomena on curved manifolds, including caustic corrections and the effects of curvature on interference patterns.